# Live inference

This notebook loads a saved bundle with `hotb_2026.inference`, classifies
windows offline and then feeds a simulated live stream through the same
predictor.

The production path rebuilds the frozen REVE encoder recorded in
`metadata.json`. To keep this demo fast and offline, the bundle is trained and
loaded with `DemoEncoder`, a deterministic stand-in that returns per-channel
RMS and zero-crossing rate. It is **not** a cognitive-load model; it only
exercises the deployment code path. Swap it for the default encoder (see the
last section) to run a real REVE bundle.


In [1]:
from pathlib import Path
import tempfile
import time

import numpy as np
from IPython.display import Markdown, display

from hotb_2026.brainaccess_connection import (
    MONTAGE,
    EEGStreamer,
    SimulatedDriver,
)
from hotb_2026.classification.dataset import (
    build_dataset,
    make_synthetic_recording,
)
from hotb_2026.classification.training import TrainingConfig, train
from hotb_2026.inference import Prediction, Predictor
from hotb_2026.model_store import StoreConfig, save_model

np.set_printoptions(precision=3, suppress=True)
workdir = Path(tempfile.mkdtemp(prefix="hotb-inference-"))
print("workdir:", workdir)


workdir: /var/folders/8d/gk38hstj1gg1tk319gmlzgm00000gn/T/hotb-inference-62lnmc_1


/Users/dominikmika/PycharmProjects/HOTB_2026/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Build a tiny bundle

`DemoEncoder` maps each 2-second window to 16 features: per-channel RMS and
zero-crossing rate for the eight montage electrodes. `build_dataset` records the
preprocessing snapshot, `train` fits the calibrated SVM, and `save_model` writes
the same `model.joblib` / `metadata.json` / `model_card.md` layout as a real
training run.


In [2]:
class DemoEncoder:
    """Fast, deterministic stand-in for the frozen REVE encoder."""

    def __init__(self, electrodes) -> None:
        self.electrodes = tuple(electrodes)
        self.model_name = "demo-rms-zcr"
        self.pos_model_name = "demo-positions"
        self.revision = "demo-v1"
        self.patch_size = 10
        self.embedding_dim = 2 * len(self.electrodes)
        self.loaded = False

    def load(self) -> None:
        self.loaded = True

    def encode(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=np.float32)
        if X.ndim == 2:
            X = X[None, ...]
        rms = np.sqrt((X ** 2).mean(axis=-1))
        sign = np.signbit(X)
        zcr = (sign[..., 1:] != sign[..., :-1]).mean(axis=-1)
        return np.concatenate([rms, zcr], axis=1).astype(np.float32)


recordings = [
    make_synthetic_recording(
        duration_s=60.0,
        state_duration_s=15.0,
        seed=seed,
        session_id=f"inference-demo-{seed + 1}",
    )
    for seed in range(6)
]
dataset = build_dataset(recordings, window_s=2.0, group_by="auto")

encoder = DemoEncoder(dataset.electrodes)
result = train(
    dataset,
    config=TrainingConfig(n_splits=3, n_trials=4, seed=0),
    encoder=encoder,
)
bundle = save_model(
    result,
    workdir / "bundle",
    config=StoreConfig(version="0.1.0"),
)

print("windows:", dataset.n_windows, "| groups:", len(set(dataset.groups)))
print("classes:", dataset.label_names)
print("CV balanced accuracy:", round(result["cv"]["mean"], 3))
print("holdout balanced accuracy:", round(result["holdout"]["metrics"]["balanced_accuracy"], 3))
print("bundle:", bundle.directory)
for path in bundle.files:
    print(f"  {path.name:20} {path.stat().st_size / 1024:7.1f} KiB")


windows: 168 | groups: 6
classes: ('low', 'medium', 'high')
CV balanced accuracy: 0.96
holdout balanced accuracy: 0.952
bundle: /var/folders/8d/gk38hstj1gg1tk319gmlzgm00000gn/T/hotb-inference-62lnmc_1/bundle
  metadata.json            3.8 KiB
  model.joblib             3.1 KiB
  model_card.md            1.7 KiB


## 2. Load with `Predictor`

`Predictor.from_bundle` validates `metadata.json` against the serialized model,
rebuilds the recorded preprocessing and exposes the input contract. Here we
inject the same `DemoEncoder`; a real bundle needs no `encoder=` argument.


In [3]:
predictor = Predictor.from_bundle(
    bundle.directory,
    encoder=DemoEncoder(dataset.electrodes),
)

print(predictor)
print("model version :", predictor.model_version)
print("window        :", predictor.window_s, "s ->", predictor.n_samples, "samples")
print("raw window    :", predictor.raw_samples, "samples at", predictor.raw_sfreq, "Hz")
print("electrodes    :", list(predictor.electrodes))
print("classes       :", list(predictor.label_names))
display(Markdown(
    "| field | value |\n"
    "|---|---|\n"
    f"| encoder | `{predictor.contract.encoder_model_name}` |\n"
    f"| revision | `{predictor.contract.encoder_revision}` |\n"
    f"| embedding dim | {predictor.contract.embedding_dim} |\n"
    f"| preprocessing output rate | {predictor.sfreq:g} Hz |"
))


Predictor(model_version='0.1.0', window_s=2, electrodes=['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2'])
model version : 0.1.0
window        : 2.0 s -> 400 samples
raw window    : 500 samples at 250.0 Hz
electrodes    : ['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2']
classes       : ['low', 'medium', 'high']


| field | value |
|---|---|
| encoder | `demo-rms-zcr` |
| revision | `demo-v1` |
| embedding dim | 16 |
| preprocessing output rate | 200 Hz |

## 3. Offline predictions

`predict_raw` accepts a `(channels, samples)` raw window in the live channel
order, keeps the trailing `window_s * raw_sfreq` samples, applies the recorded
preprocessing once and returns a `Prediction` dataclass.

`current_time` is the stream time at the **end** of the window.
`state_change_time` is when the current *predicted* label was first reported -
live inference has no ground truth.


In [4]:
recording = recordings[0]
rows = []

for start_s in (0.0, 15.0, 30.0, 45.0):
    start = int(start_s * predictor.raw_sfreq)
    raw_window = recording.signal[:, start : start + predictor.raw_samples]
    prediction = predictor.predict_raw(
        raw_window,
        current_time=start_s + predictor.window_s,
    )
    rows.append(prediction)

for prediction in rows:
    probabilities = "  ".join(
        f"{name}={value:.2f}"
        for name, value in prediction.probabilities.items()
    )
    print(
        f"{prediction.current_time:5.1f}s  {prediction.label_name:>6}  "
        f"p={prediction.confidence:.2f}  "
        f"state since {prediction.state_change_time:5.1f}s  | {probabilities}"
    )

print()
print("window:", rows[0].window_start, "->", rows[0].window_end, "s")
print("model version:", rows[0].model_version)
assert all(isinstance(row, Prediction) for row in rows)


  2.0s     low  p=0.99  state since   2.0s  | low=0.99  medium=0.01  high=0.00
 17.0s  medium  p=0.90  state since  17.0s  | low=0.10  medium=0.90  high=0.00
 32.0s    high  p=0.96  state since  32.0s  | low=0.00  medium=0.04  high=0.96
 47.0s     low  p=0.97  state since  47.0s  | low=0.97  medium=0.03  high=0.00

window: 0.0 -> 2.0 s
model version: 0.1.0


## 4. Simulated live stream

`Predictor.processor()` returns a `WindowedPreprocessor` for `EEGStreamer`. Give
the streamer a window **longer** than the model window (`window_s + 1.0` here) so
the adapter slices exact, non-overlapping windows out of its ring buffer.
`warmup()` loads the encoder before the first window arrives.


In [5]:
predictions: list[Prediction] = []
predictor.reset()

streamer = EEGStreamer(
    SimulatedDriver(MONTAGE, chunk_ms=40.0, seed=0),
    MONTAGE,
    stream_name="HOTB inference demo",
    source_id="hotb-inference-demo",
    window_s=predictor.window_s + 1.0,
)
streamer.add_processor(
    predictor.processor(on_prediction=predictions.append)
)
predictor.warmup()

streamer.start()
try:
    deadline = time.monotonic() + 25.0
    while (
        streamer.running
        and len(predictions) < 3
        and time.monotonic() < deadline
    ):
        time.sleep(0.05)
finally:
    streamer.stop()

print("streamer error:", streamer.error)
for prediction in predictions:
    print(
        f"{prediction.current_time:5.2f}s  {prediction.label_name:>6}  "
        f"p={prediction.confidence:.2f}  "
        f"state since {prediction.state_change_time:5.2f}s"
    )
assert predictions, "no windows were classified"


2026-11-15 09:10:48.784 (   0.002s) [          109A0A]         api_config.cpp:128   INFO| Loaded default config
2026-11-15 09:10:48.784 (   0.003s) [          109A0A]             common.cpp:82    INFO| git:a497f3c36d6cc0938fc46a850b17a82c478b20bc/branch:refs/tags/v1.18.0.b7/build:Release/compiler:AppleClang-21.0.0.21000101/link:SHARED


streamer error: None
 2.00s    high  p=0.98  state since  2.00s
 4.00s    high  p=0.98  state since  2.00s
 6.00s    high  p=0.98  state since  2.00s


## 5. Production and CLI

A real bundle records the REVE model ids and revision, so the default encoder is
correct and no override is needed:

```python
predictor = Predictor.from_bundle("bundles/nback")
predictor = Predictor.from_pretrained(
    "ai-med/hotb-2026-nback-classifier",
    revision="0.1.0",  # tag, branch or commit
)
```

Run the simulated CLI (one JSON `Prediction` per completed window):

```bash
uv run python -m hotb_2026.inference --bundle bundles/nback --windows 3
uv run python -m hotb_2026.inference \
    --repo-id ai-med/hotb-2026-nback-classifier --revision 0.1.0
```

For a live headset, pass `MONTAGE` (or the recording's electrode list) as
`electrodes=` when the channel order differs from training. Extra channels are
ignored; a missing training electrode fails at startup.

`joblib.load` executes arbitrary code, so only load bundles you trust. See
[`src/hotb_2026/inference/README.md`](../../src/hotb_2026/inference/README.md)
for the full contract and validation rules.
